# Tree recovery on the runner — jobs C (T1), B (8a + Jensen–Shannon), A (T4)

**Run all cells.** Every job is resumable: re-running a cell is always safe, and a scoring loop picks up where it
left off after a runtime disconnect. One runtime runs **one job** (`JOB`); for job B, one runtime scores one
model (`MODEL_KEY`) and the record is assembled from whichever per-model caches exist.

| job | what | model | send back |
|---|---|---|---|
| **C** | T1: L2 and Jensen–Shannon distance between the next-word states at every span's two ends | Qwen3-0.6B-Base, float32 | `tree_t1_qwen3_0.6b.jsonl` |
| **B** | 8a again, adding `js_ret`, `js_ctrl`, `head_ret`, `head_ctrl` and repeating `ret`, `ctrl` | the five models of the 8a run, float32 | `record_8a_js.json` |
| **A** | T4: every substituted sentence of the 1b cache scored token by token, plus its own end mark | Qwen3-0.6B-Base, float32 | `tree_t4_qwen3_0.6b.jsonl.gz` |

Suggested order: **C, then B, then A** (smallest first). The computation is the package's reference code
`tree_runner_ref.py`, imported **unmodified** (its sha256 is checked and written into every meta block).

**Every job has a check on its first sentences, and the full run refuses to start until it passes:**

- C — against `data/anchor_tree_local.json` (20 sentences, 3,505 spans): Spearman ≥ 0.999 and median relative
  difference ≤ 1 % for both distances.
- B — `ret`/`ctrl` on the first frames must match the delivered `record_8a_multimodel.json` within 2 % relative
  (and again on every row when the record is built).
- A — (1) the anchor's two sentences: token counts identical, `total`/`suf_sub`/`end_sub` within 0.05 nats;
  (2) the first 50 sentences against the 1b cache: `n_tok` equal on every row, `total` Spearman ≥ 0.999;
  (3) causality: `|pre_check|` ≤ 1e-3 on every row.

**Inputs you provide** (on Drive, or uploaded to `OUTPUT_DIR`): the 1b span-cost cache of Qwen3-0.6B-Base (jobs A
and C), `frames_8a_seed8.json` and `record_8a_multimodel.json` (job B).

## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

**This is the only cell you normally edit.** Pick `JOB`. For job B also pick `MODEL_KEY` (one per runtime, or
`all-small` for the three small models in one go). Input files are looked up in `OUTPUT_DIR` first, then in
`DRIVE_DIR` (copied over when found there). Jobs A and C load Qwen3-0.6B-Base at the pinned revision in
**float32** (a deliberate change from the 1b run's bfloat16); job B keeps the 8a run's float32.

In [ ]:
#@title Run configuration { display-mode: "form" }
JOB              = "C"            #@param ["C", "B", "A"]
# --- jobs A and C ---
SPAN_CACHE_FILE  = "span_costs_Qwen-Qwen3-0.6B-Base_min-over-it-there-did-then-do-so-does-so-did-so-done-so-doing-so-is-was-be-been-happens-happened-controls-blorp-del.jsonl"  #@param {type:"string"}
TREE_ANCHOR_FILE = "data/anchor_tree_local.json"  #@param {type:"string"}
T4_BATCH         = 128            #@param {type:"integer"}
T4_CHECK_FIRST   = 50             #@param {type:"integer"}
# --- job B ---
MODEL_KEY        = "qwen25_0.5b"  #@param ["qwen25_0.5b", "qwen3_0.6b", "qwen3_1.7b", "qwen3_8b", "llama31_8b", "all-small"]
FRAMES_FILE      = "frames_8a_seed8.json"         #@param {type:"string"}
RECORD_8A_FILE   = "record_8a_multimodel.json"    #@param {type:"string"}
FRAMES_PER_BATCH = 8              #@param {type:"integer"}
B_CHECK_FRAMES   = 10             #@param {type:"integer"}
REVISION_OVERRIDES = "{}"         #@param {type:"string"}
# --- run mechanics ---
LIMIT            = 0              #@param {type:"integer"}
MIRROR_EVERY     = 10             #@param {type:"integer"}
MIRROR_TO_DRIVE  = True           #@param {type:"boolean"}
DRIVE_DIR        = "/content/drive/MyDrive/m1_llms_analyzer/tree_runner"  #@param {type:"string"}
OUTPUT_DIR       = "outputs/tree_runner"          #@param {type:"string"}

import json
import shutil
from pathlib import Path

from m1_analyzer import ModelConfig, RunConfig, ScoringConfig, StorageConfig, in_colab
from m1_analyzer.experiments import CODES_8A, MODELS_8A, SMALL_KEYS, tree_runner
from m1_analyzer.experiments.task_8a_js import MODELS_8A_JS

assert JOB in ("A", "B", "C"), JOB
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

SPAN_CACHE = OUT / SPAN_CACHE_FILE
TREE_ANCHOR = Path(TREE_ANCHOR_FILE)
WORK = {"C": OUT / "work_t1.jsonl", "A": OUT / "work_t4.jsonl"}             # resumable working caches
CHECKS = {"C": OUT / "checks_t1.json", "A": OUT / "checks_t4.json"}         # the check numbers, for the meta block
DELIVERABLE = {"C": OUT / tree_runner.T1_OUTPUT, "A": OUT / tree_runner.T4_OUTPUT, "B": OUT / "record_8a_js.json"}

FRAMES_PATH = OUT / FRAMES_FILE
RECORD_8A = OUT / RECORD_8A_FILE
KEYS = [k for k in SMALL_KEYS if k in MODELS_8A_JS] if MODEL_KEY == "all-small" else [MODEL_KEY]
_revisions = json.loads(REVISION_OVERRIDES or "{}")
SPECS = {k: MODELS_8A[k].with_overrides(revision=_revisions.get(k, MODELS_8A[k].revision)) for k in MODELS_8A_JS}


def js_cache_for(key: str) -> Path:
    return OUT / f"scores_8a_js_{key}.jsonl"


def make_config_b(spec) -> RunConfig:
    """Exactly the 8a run: float32 weights, LM head, no extra BOS (tok(prefix) carries the tokenizer's own)."""
    return RunConfig(
        model=ModelConfig(model_id=spec.hf_id, revision=spec.revision, head="causal_lm", dtype="float32",
                          trust_remote_code=spec.trust_remote_code),
        scoring=ScoringConfig(bos_policy="none", batch_size=len(CODES_8A) * FRAMES_PER_BATCH, max_length_cap=512),
        storage=StorageConfig(output_dir=str(OUT)),
    )


def drive_dir():
    """The Drive folder, mounted on first use; None outside Colab or when mirroring is off."""
    if not (MIRROR_TO_DRIVE and in_colab()):
        return None
    target = Path(DRIVE_DIR)
    if not Path("/content/drive").exists():
        from google.colab import drive  # noqa: F401 -- Colab only
        drive.mount("/content/drive")
    target.mkdir(parents=True, exist_ok=True)
    return target


def restore(path: Path) -> bool:
    """Copy `path` back from Drive when it is missing locally. True if it is now present."""
    if path.exists():
        return True
    drive = drive_dir()
    if drive is not None and (drive / path.name).exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(drive / path.name, path)
        print(f"restored {path.name} from {drive}")
    return path.exists()


def mirror_path(path: Path):
    drive = drive_dir()
    return None if drive is None else drive / path.name


def save_json(path: Path, payload) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(payload, indent=1), encoding="utf-8")
    tmp.replace(path)
    if mirror_path(path):
        shutil.copy(path, mirror_path(path))


print(f"job                 : {JOB}")
print(f"reference code      : sha256 {tree_runner.reference_sha256()[:12]}…  unmodified={tree_runner.reference_unmodified()}")
if not tree_runner.reference_unmodified():
    raise SystemExit("tree_runner_ref.py differs from the package's; restore it before running any job.")
if JOB in ("A", "C"):
    print(f"model               : {tree_runner.MODEL_ID} @ {str(tree_runner.MODEL_REVISION)[:12]}  float32")
    print(f"1b cache            : {SPAN_CACHE}")
    print(f"anchor              : {TREE_ANCHOR}")
    print(f"deliverable         : {DELIVERABLE[JOB]}")
else:
    print(f"models this runtime : {KEYS}")
    print(f"frames / 8a record  : {FRAMES_PATH} / {RECORD_8A}")
    print(f"deliverable         : {DELIVERABLE['B']}")
print("Configuration ready.")

## 5 · Smoke test (no download, a few seconds)

Runs jobs C, A and B end to end on a random 4-layer GPT-2 with a word-level vocabulary: T1 and T4 on three short
sentences (resumed, checked against themselves, finalised), and the 8a passes with Jensen–Shannon on three
frames, matched against a record built from the 8a scorer. The numbers are meaningless; this proves the plumbing.

In [ ]:
#@title Smoke test on a tiny model
import tempfile
import time

from m1_analyzer import Analyzer
from m1_analyzer.experiments import FRAME_WORDS_8A, ModelSpec, build_record_8a, generate_frames_8a, save_frames_8a
from m1_analyzer.experiments import score_model_8a, task_8a_js
from m1_analyzer.experiments import tree_runner_ref as ref
from m1_analyzer.testing import build_tiny_local_model

started = time.perf_counter()
words = [["the", "quick", "brown", "fox", "jumps"], ["a", "lazy", "dog", "one", "two"]]
trees = ["(S (NP (DT the) (JJ quick) (JJ brown) (NN fox)) (VP (VBZ jumps)) (. .))",
         "(S (NP (DT a) (JJ lazy) (NN dog)) (NP (CD one) (CD two)) (. ?))"]
fill = sorted({w for f in ref.FILLERS if f != "<del>" for w in f.split()})
extra = fill + [w[:1].upper() + w[1:] for w in fill + sum(words, [])] + ["<|endoftext|>"]
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    path = build_tiny_local_model(tmp / "tiny", extra_vocab=extra, max_positions=64)
    tok, mdl, bos, device, smoke = tree_runner.load_model_f32(path, revision=None, device="cpu")
    sents = [{"id": f"s{k}", "words": w, "tree": t,
              "keys": [f"{i},{j}" for i in range(len(w)) for j in range(i + 1, len(w)) if (i, j) != (0, len(w) - 1)]}
             for k, (w, t) in enumerate(zip(words, trees))]
    t1 = tree_runner.run_t1(tok, mdl, bos, sents, cache_path=tmp / "t1.jsonl", show_progress=False)
    assert tree_runner.run_t1(tok, mdl, bos, sents, cache_path=tmp / "t1.jsonl", show_progress=False) == t1
    assert tree_runner.check_t1_anchor(t1, {"sentences": [{"id": r["id"], "t1": r["t1"]} for r in t1]})["pass"]
    t4 = tree_runner.run_t4(tok, mdl, bos, sents, batch=8, cache_path=tmp / "t4.jsonl", show_progress=False)
    assert tree_runner.check_t4_anchor(t4, t4)["pass"] and tree_runner.check_t4_causality(t4)["pass"]
    tree_runner.finalize(t4, tmp / "t4.jsonl.gz", tree_runner.run_meta("t4", model_id="tiny", revision=None), job="t4")
    smoke.unload()

    path = build_tiny_local_model(tmp / "tiny8a", extra_vocab=FRAME_WORDS_8A, max_positions=64)
    smoke = Analyzer(RunConfig(model=ModelConfig(model_id=path, head="causal_lm", dtype="float32"),
                               scoring=ScoringConfig(bos_policy="none", batch_size=64)))
    spec = ModelSpec("tiny", path)
    frames = save_frames_8a(generate_frames_8a({"tiny": smoke.models.tokenizer}, n=3, seed=8), tmp / "frames.json")
    score_model_8a(smoke, spec, frames, cache_path=tmp / "s8a.jsonl", show_progress=False)
    rec8a = build_record_8a(frames, {"tiny": tmp / "s8a.jsonl"}, keys=["tiny"])
    task_8a_js.score_model_8a_js(smoke, spec, frames, cache_path=tmp / "js.jsonl", show_progress=False)
    rec = task_8a_js.build_record_8a_js({"tiny": tmp / "js.jsonl"}, rec8a, frames)
    smoke.unload()
print(f"T1: {sum(len(r['t1']) for r in t1)} spans; T4: {sum(len(t) for r in t4 for t in r['rows'].values())} rows; "
      f"8a+JS: {len(rec['models']['tiny']['rows'])} rows, L2 match {rec['meta']['l2_match_max_rel']}; "
      f"{time.perf_counter() - started:.1f}s")
print("\nSmoke test passed (the numbers are meaningless: a random 5 MB model).")

## 6 · Jobs A and C — inputs and model

Reads the 1b cache: `words` and `tree` of every sentence, the span inventory (the keys of `spans["it"]`,
173,966 spans in all) and the header's 17 fillers (`<del>` included; refused if they differ from the
reference's). Then loads Qwen3-0.6B-Base at the pinned revision in float32, with BOS `<|endoftext|>` as an id.
Skipped for job B.

In [ ]:
#@title Load the 1b cache, the anchor and the model (jobs A and C)
import time

if JOB == "B":
    print("Job B: nothing to load here; go to section 10.")
else:
    if not restore(SPAN_CACHE):
        raise SystemExit(f"The 1b cache is not at {SPAN_CACHE} (nor in {DRIVE_DIR}). Upload it there and re-run.")
    span_header, SENTENCES = tree_runner.load_1b_cache(SPAN_CACHE)
    ANCHOR = tree_runner.load_anchor(TREE_ANCHOR)
    print(f"1b cache : {len(SENTENCES)} sentences, {tree_runner.count_spans(SENTENCES):,} spans, "
          f"{len(span_header['proforms'])} fillers; header model {span_header.get('model_id')}")
    started = time.perf_counter()
    tok, mdl, bos, device, analyzer = tree_runner.load_model_f32(tree_runner.MODEL_ID, tree_runner.MODEL_REVISION)
    MODEL_META = {"model_id": tree_runner.MODEL_ID, "revision": analyzer.models.metadata()["revision"],
                  "dtype": analyzer.models.metadata()["dtype"], "bos_id": bos}
    print(f"model    : {MODEL_META['model_id']} @ {str(MODEL_META['revision'])[:12]}  weights {MODEL_META['dtype']}  "
          f"device {device}  BOS id {bos}  ({time.perf_counter() - started:.1f}s)")
    if MODEL_META["dtype"] != "float32":
        raise SystemExit("Jobs A and C run in float32; the model loaded in another dtype.")
    first = SENTENCES[0]
    print(f"\nfirst sentence {first['id']}: {tree_runner.ref.text_of(first['words'])!r}")
    print(f"  end mark {tree_runner.ref.end_string(first['tree'])!r}, {len(first['keys'])} spans")

## 7 · The check on the first sentences — look at it before the full job

- **Job C**: the 20 anchor sentences are scored (they go into the working cache, so the full run does not redo
  them) and compared with the anchor, per distance.
- **Job A**: the two anchor sentences and the first `T4_CHECK_FIRST` (50) sentences are scored; check 1 against
  the anchor, check 2 against the 1b cache, check 3 on causality.

The numbers are saved to `checks_t1.json` / `checks_t4.json` and go into the deliverable's meta block. **If a
check fails the cell stops: do not run the full job; report the numbers instead.**

In [ ]:
#@title Run the check
import time

def show(name, block):
    print(f"\n{name}: {'PASS' if block['pass'] else 'FAIL'}")
    for key, value in block.items():
        if key in ("pass",) or (isinstance(value, list) and not value):
            continue
        print(f"  {key:<24} {value}")

if JOB == "B":
    print("Job B checks its L2 against record_8a_multimodel.json in section 10.")
else:
    restore(WORK[JOB])
    started = time.perf_counter()
    if JOB == "C":
        ids = [s["id"] for s in ANCHOR["sentences"]]
        rows = tree_runner.run_t1(tok, mdl, bos, SENTENCES, device=device, cache_path=WORK[JOB], model_meta=MODEL_META,
                                  input_name=SPAN_CACHE.name, ids=ids, mirror_path=mirror_path(WORK[JOB]))
        checks = {"anchor": tree_runner.check_t1_anchor(rows, ANCHOR)}
    else:
        ids = {a["id"] for a in ANCHOR["t4"]} | {s["id"] for s in SENTENCES[:T4_CHECK_FIRST]}
        rows = tree_runner.run_t4(tok, mdl, bos, SENTENCES, device=device, batch=T4_BATCH, cache_path=WORK[JOB],
                                  model_meta=MODEL_META, input_name=SPAN_CACHE.name, ids=sorted(ids),
                                  mirror_path=mirror_path(WORK[JOB]))
        first = [r for r in rows if r["id"] in {s["id"] for s in SENTENCES[:T4_CHECK_FIRST]}]
        checks = {
            "anchor": tree_runner.check_t4_anchor(rows, ANCHOR["t4"]),
            "cache": tree_runner.check_t4_cache(first, SENTENCES),
            "causality": tree_runner.check_t4_causality(rows),
        }
        seconds = tree_runner.wall_seconds(rows)
        n_sub = sum(len(t) for r in rows for t in r["rows"].values())
        total_sub = tree_runner.count_spans(SENTENCES) * len(tree_runner.ref.FILLERS)
        print(f"throughput: {seconds / max(n_sub, 1) * 1000:.2f} ms per substituted sentence at batch {T4_BATCH} "
              f"-> about {seconds / max(n_sub, 1) * total_sub / 3600:.1f} h for all {total_sub:,}")
    checks["pass"] = all(b["pass"] for b in checks.values() if isinstance(b, dict))
    checks["batch"] = T4_BATCH if JOB == "A" else None
    save_json(CHECKS[JOB], checks)
    for name, block in checks.items():
        if isinstance(block, dict):
            show(name, {k: v for k, v in block.items() if k not in ("count_mismatch_examples",) or v})
    print(f"\n{len(rows)} sentences in {time.perf_counter() - started:.0f}s; saved to {CHECKS[JOB]}")
    if not checks["pass"]:
        raise SystemExit("A check FAILED. Stop here and send the numbers above (and checks_*.json) back.")
    print("\nAll checks pass. Look at the numbers above, then run the full job.")

## 8 · The full job (GPU, resumable)

Continues the working cache (one fsynced line per sentence; the check's sentences are already there). A
disconnect loses at most one sentence: re-run this cell and it picks up where it stopped. `LIMIT` > 0 scores
only that many sentences (a dry run). Job A's `T4_BATCH` changes only how many substitutions share a forward
pass; beyond float rounding it changes no number (check 1 shows that).

In [ ]:
#@title Run the full job (A or C)
import time

if JOB == "B":
    print("Job B runs in section 10.")
else:
    checks = json.loads(CHECKS[JOB].read_text(encoding="utf-8")) if restore(CHECKS[JOB]) else None
    if not checks or not checks.get("pass"):
        raise SystemExit("Run section 7 and pass its check before the full job.")
    started = time.perf_counter()
    kwargs = dict(device=device, cache_path=WORK[JOB], model_meta=MODEL_META, input_name=SPAN_CACHE.name,
                  limit=LIMIT or None, mirror_path=mirror_path(WORK[JOB]), mirror_every=MIRROR_EVERY)
    if JOB == "C":
        ROWS = tree_runner.run_t1(tok, mdl, bos, SENTENCES, **kwargs)
    else:
        ROWS = tree_runner.run_t4(tok, mdl, bos, SENTENCES, batch=T4_BATCH, **kwargs)
    print(f"{len(ROWS)} / {len(SENTENCES)} sentences in {WORK[JOB]} ({time.perf_counter() - started:.0f}s this session, "
          f"{tree_runner.wall_seconds(ROWS) / 3600:.2f} h of compute in all)")

## 9 · The deliverable (A or C)

Writes `tree_t1_qwen3_0.6b.jsonl` (C) or `tree_t4_qwen3_0.6b.jsonl.gz` (A): first line the meta block — model
id and revision, dtype, torch and transformers versions, GPU, wall time (summed per-sentence compute),
the reference code's sha256 and whether it was modified, and the check numbers — then one line per sentence,
`{"id", "t1"}` or `{"id", "end", "orig", "rows"}`, in cache order. Refuses a partial run.

In [ ]:
#@title Write the deliverable
if JOB == "B":
    print("Job B's record is written in section 11.")
else:
    _, ROWS = tree_runner.load_rows(WORK[JOB])
    order = {s["id"]: k for k, s in enumerate(SENTENCES)}
    ROWS = sorted(ROWS, key=lambda r: order[r["id"]])
    checks = json.loads(CHECKS[JOB].read_text(encoding="utf-8"))
    meta = tree_runner.run_meta(
        {"C": "t1", "A": "t4"}[JOB], model_id=MODEL_META["model_id"], revision=MODEL_META["revision"],
        dtype=MODEL_META["dtype"], wall_seconds=tree_runner.wall_seconds(ROWS), checks=checks,
        extra={"input_cache": SPAN_CACHE.name, "n_sentences": len(ROWS), "n_spans": tree_runner.count_spans(SENTENCES),
               "fillers": list(tree_runner.ref.FILLERS), "bos_id": bos,
               "batch": T4_BATCH if JOB == "A" else None, "notebook": "experiment_tree.ipynb"},
    )
    path = tree_runner.finalize(ROWS, DELIVERABLE[JOB], meta, job=meta["job"], expected=len(SENTENCES))
    if mirror_path(path):
        shutil.copy(path, mirror_path(path))
    print(f"{path} ({path.stat().st_size / 1024 / 1024:.1f} MB)")
    print(json.dumps({k: v for k, v in meta.items() if k != "checks"}, indent=1))

## 10 · Job B — the 8a passes with Jensen–Shannon (one model per runtime)

Exactly the 8a pipeline (your `frames_8a_seed8.json`, float32 weights and logits, `tok(prefix)` with the
tokenizer's default special tokens, the state at the last token) with four values added per row. **The check:**
the first `B_CHECK_FRAMES` frames are scored first and their `ret`/`ctrl` compared with the delivered
`record_8a_multimodel.json`; over 2 % relative stops the cell. Then the remaining frames, resumable.

In [ ]:
#@title Score this runtime's model(s) for job B (resumable)
import time

from m1_analyzer import Analyzer
from m1_analyzer.experiments import task_8a_js

if JOB != "B":
    print(f"Job {JOB}: nothing to do here.")
else:
    for path in (FRAMES_PATH, RECORD_8A):
        if not restore(path):
            raise SystemExit(f"{path.name} is not in {OUT} (nor in {DRIVE_DIR}). Upload the file from your 8a run.")
    record_8a = task_8a_js.load_record_8a(RECORD_8A)
    B_CHECKS = json.loads((OUT / "checks_8a_js.json").read_text()) if restore(OUT / "checks_8a_js.json") else {}
    analyzer = None
    for key in KEYS:
        spec = SPECS[key]
        cache = js_cache_for(key)
        restore(cache)
        if analyzer is not None:
            analyzer.unload()
        started = time.perf_counter()
        analyzer = Analyzer(make_config_b(spec))
        print(f"Loaded {key} ({spec.hf_id}) in {time.perf_counter() - started:.1f}s; "
              f"weights {analyzer.models.metadata()['dtype']}")
        common = dict(cache_path=cache, frames_per_batch=FRAMES_PER_BATCH, show_progress=True,
                      mirror_path=mirror_path(cache), mirror_every=MIRROR_EVERY,
                      provenance={"notebook": "experiment_tree.ipynb", "job": "B"})
        _, rows = task_8a_js.score_model_8a_js(analyzer, spec, FRAMES_PATH, limit=B_CHECK_FRAMES, **common)
        check = task_8a_js.l2_match(task_8a_js.rows_from_cache(rows), record_8a, key)
        B_CHECKS[key] = check
        save_json(OUT / "checks_8a_js.json", B_CHECKS)
        print(f"  check on {check['n_rows']} rows: max rel L2 diff {check['max_rel']:.2e} (tol {check['tol']}) "
              f"-> {'PASS' if check['pass'] else 'FAIL'}; worst {check['worst']}")
        if not check["pass"]:
            raise SystemExit(f"{key}: L2 does not match the 8a record within 2 %. Stop and report.")
        started = time.perf_counter()
        header, rows = task_8a_js.score_model_8a_js(analyzer, spec, FRAMES_PATH, limit=LIMIT or None, **common)
        flat = task_8a_js.rows_from_cache(rows)
        full = task_8a_js.l2_match(flat, record_8a, key)
        heads = sorted(r["head_ret"] for r in flat)
        print(f"  {key}: {len(rows)} / {header['n_frames']} frames ({time.perf_counter() - started:.0f}s); "
              f"L2 max rel diff {full['max_rel']:.2e}; median head share (ret) {heads[len(heads) // 2]:.4f}")

## 11 · Job B — the record (CPU, from whichever caches exist)

Merges `scores_8a_js_<key>.jsonl` of the five models into `record_8a_js.json` in the requested schema:
`meta` (date, reference sha, `l2_match_max_rel` per model, versions) and `models{key: {rows: [...]}}` with
`frame, structure, control, ret, ctrl, js_ret, js_ctrl, head_ret, head_ctrl` per row. The build re-checks the
L2 match on every row and refuses a model over 2 %.

In [ ]:
#@title Assemble record_8a_js.json
if JOB != "B":
    print(f"Job {JOB}: nothing to do here.")
else:
    caches = {k: js_cache_for(k) for k in MODELS_8A_JS if restore(js_cache_for(k))}
    print(f"caches found : {sorted(caches)}; missing: {[k for k in MODELS_8A_JS if k not in caches] or 'none'}")
    record = task_8a_js.build_record_8a_js(caches, RECORD_8A, FRAMES_PATH, allow_partial=True)
    save_json(DELIVERABLE["B"], record)
    print(f"{DELIVERABLE['B']}: models {list(record['models'])}")
    print(json.dumps({k: record["meta"][k] for k in ("date", "reference_code_sha", "l2_match_max_rel", "versions")}, indent=1))

## 12 · Persist and download

In [ ]:
#@title Copy every output to Google Drive (survives a runtime disconnect)
from m1_analyzer import in_colab

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DIR)
    target.mkdir(parents=True, exist_ok=True)
    outputs = list(WORK.values()) + list(CHECKS.values()) + list(DELIVERABLE.values()) + [OUT / "checks_8a_js.json"]
    outputs += [js_cache_for(k) for k in MODELS_8A_JS]
    for path in outputs:
        if path.exists():
            shutil.copy(path, target / path.name)
            print("copied", target / path.name)
else:
    print("Not running in Colab -- skipping Drive mount.")

In [ ]:
#@title Download this job's deliverable
from m1_analyzer import in_colab

path = DELIVERABLE[JOB]
if in_colab() and path.exists():
    from google.colab import files
    files.download(str(path))
else:
    print("Deliverable:", path, "(exists)" if path.exists() else "(not written yet)")

In [ ]:
#@title Run the test suite inside Colab
import subprocess, sys

result_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_tree_runner.py"], capture_output=True, text=True
)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `tree_runner_ref.py differs from the package's` | Someone edited the reference. `git checkout src/m1_analyzer/experiments/tree_runner_ref.py`; the jobs must run it unmodified. |
| `The 1b cache is not at …` | Upload the 1b span-cost cache to `OUTPUT_DIR` or `DRIVE_DIR` under `SPAN_CACHE_FILE`. |
| `proforms … are not the reference's fillers` | This is not the 17-filler cache of the 1b run. Point `SPAN_CACHE_FILE` at the right file. |
| `was written for dtype=…` on resume | The working cache belongs to another setting. Use another `OUTPUT_DIR`, or delete `work_t*.jsonl`. |
| A check fails | Stop. Send `checks_t1.json` / `checks_t4.json` / `checks_8a_js.json` back; do not run the full job. |
| CUDA out of memory in job A | Lower `T4_BATCH` (64 is the reference's default). It changes no number. |
| `Access to 'meta-llama/…' was denied` (job B) | Accept the licence on the model page and add `HF_TOKEN` as a Colab secret with notebook access. |